# 4주차 과제 베이스라인 — 물류 유통량 예측

송하인·수하인의 격자공간 ID와 물품 카테고리로 운송장 건수를 예측합니다. 이 베이스라인은 **원본 범주형 변수와 학습 데이터에만 맞추는 one-hot Pipeline**으로 Random Forest 한 개를 실행하는 가장 짧은 경로를 제공합니다. 높은 점수나 완성된 분석보다 직접 실행한 내용, 오류를 나눈 과정, 다음 행동을 기록하는 일이 더 중요합니다. `train.csv`가 없으면 명시적인 **연습용 소규모 예시 데이터**로 전환되며, 그 결과는 실제 물류 데이터에 대한 결론으로 사용하지 않습니다.


## 1. 데이터 불러오기

원본 데이터와 변수 설명은 [데이콘 물류 유통량 예측 경진대회](https://dacon.io/competitions/official/235867) 페이지에서 확인하고 내려받을 수 있습니다. 페이지에 로그인한 뒤 데이터 탭의 안내에 따라 내려받습니다. 압축을 푼 파일을 아래 코드에 적힌 경로에 둡니다. 다음 셀은 파일 상태를 알려 주고, 기본 파일이 없으면 연습용 데이터로 실행 흐름을 이어 갑니다. competition `test.csv`는 선택 제출 파일이므로 없어도 기본 시도를 진행합니다.


In [ ]:
from pathlib import Path

import pandas as pd

data_dir = Path('../dataset/extracted/물류 유통량 예측 경진대회')
train_path = data_dir / 'train.csv'
test_path = data_dir / 'test.csv'
print('현재 작업 폴더:', Path.cwd())
missing_files = [path.name for path in [train_path, test_path] if not path.exists()]
print('누락 파일:', missing_files if missing_files else '없음')

if train_path.exists():
    DATA_MODE = '실제 데이터'
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path) if test_path.exists() else None
else:
    DATA_MODE = '연습용 소규모 예시 데이터'
    categories = ['식품', '생활용품', '의류', '전자기기']
    demo_rows = []
    for i in range(60):
        sender = f'DEMO_S_{i % 6}'
        receiver = f'DEMO_R_{(i * 3) % 8}'
        category = categories[i % len(categories)]
        volume = 2 + (i % 6) + (i % len(categories)) * 3 + ((i * 3) % 8)
        demo_rows.append({
            '송하인_격자공간고유번호': sender,
            '수하인_격자공간고유번호': receiver,
            '물품_카테고리': category,
            '운송장_건수': volume,
        })
    train = pd.DataFrame(demo_rows)
    test = None
    print('train.csv가 없어 예시 데이터로 코드 흐름만 연습합니다.')
    print('예시 RMSE와 예측값은 실제 물류 데이터의 결론이 아닙니다.')

print('데이터 모드:', DATA_MODE)
print('기본 시도 train 크기:', train.shape)
print('선택 제출 test 상태:', test.shape if test is not None else '파일 없음 — 기본 시도에는 필요하지 않습니다.')


In [ ]:
train.head()


## 2. 범주형 변수와 one-hot Pipeline 준비

`송하인_격자공간고유번호`와 `수하인_격자공간고유번호`는 원본이 숫자(int64)이지만, 순서에 의미가 없는 **위치 코드**입니다. 값이 크다고 운송량이 늘어나는 관계가 아니므로 문자열 범주로 다룹니다. `물품_카테고리`도 같은 방식으로 준비합니다.

아래 `make_pipeline`은 `OneHotEncoder(handle_unknown='ignore')`를 모델과 묶습니다. 학습·검증 데이터를 나눈 뒤 Pipeline을 `X_train`에 fit하면 범주 목록도 학습 데이터에서만 정해집니다. 검증이나 competition test에 처음 보는 ID가 있어도 변환 오류는 나지 않지만, **알지 못하는 ID의 운송량 패턴을 새로 학습해 주는 것은 아닙니다.**


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

cat_cols = ['송하인_격자공간고유번호', '수하인_격자공간고유번호', '물품_카테고리']

X = train[cat_cols].astype('string')
y = train['운송장_건수']

def make_pipeline(model):
    return Pipeline([
        ('encode', OneHotEncoder(handle_unknown='ignore')),
        ('model', model),
    ])

X.head()


## 이번 주 과제 경로

- **✅ 기본 시도**: 아래 셀을 순서대로 실행해 무작위 학습·검증 분할과 Random Forest 한 개의 validation RMSE를 확인합니다.
- **🧩 문제 분해**: 실행이 멈추면 마지막 성공 셀, 오류 마지막 줄, 의심 원인 하나, 작은 확인 코드 하나를 기록합니다. 해결하지 못해도 괜찮습니다.
- **🌱 선택 탐색**: 여유가 있을 때만 다른 모델 한 개, 그룹 분할, 미지 ID별 오차, permutation importance 중 하나를 골라 봅니다.

기본 완료 기준은 **질문 1개 + 실행 또는 실행 시도 1개 + 관찰 결과 또는 오류 1개 + 다음 행동 1개**입니다. competition 제출과 세 모델 비교는 기본 범위가 아닙니다.


### 3. ✅ 기본 시도 — 학습/검증 데이터 분리

이 분할은 같은 모집단에서 새로운 운송 행이 온다는 가정의 첫 시도입니다. competition `test`는 분할과 encoder 학습에 사용하지 않습니다.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('학습 크기:', X_train.shape, '/ 검증 크기:', X_valid.shape)
print('competition test를 분할에 사용하지 않았습니다.')


### 4. ✅ 기본 시도 — Random Forest 한 개 실행

처음에는 모델 하나만 끝까지 실행합니다. 아래 값은 정답 설정이 아니라 실행 시간을 줄인 출발점입니다. 실행이 오래 걸리면 `n_estimators=10`으로 낮추어도 됩니다.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

model = make_pipeline(RandomForestRegressor(
    n_estimators=40, max_depth=8, min_samples_leaf=5, max_features='sqrt',
    random_state=42, n_jobs=-1,
))
model.fit(X_train, y_train)
pred_valid = model.predict(X_valid)
rmse = mean_squared_error(y_valid, pred_valid) ** 0.5
print(f'[{DATA_MODE}] validation RMSE: {rmse:,.3f}')


### 5. 결과 또는 오류를 한 가지 관찰합니다

RMSE가 나왔다면 실제값·예측값·오차의 첫 다섯 행을 살펴보고 한 가지를 적습니다. 실행이 실패했다면 이 셀을 건너뛰고 오류 메시지의 마지막 줄과 마지막 성공 셀을 기록합니다.


In [ ]:
result = pd.DataFrame({
    '실제값': y_valid.to_numpy(),
    '예측값': pred_valid,
})
result['오차(실제-예측)'] = result['실제값'] - result['예측값']
result.head()

# 작은 힌트: 큰 오차 한 행을 골라 원본 세 범주의 빈도와 미지 ID 여부를 확인해 볼 수 있습니다.


## 🌱 선택 탐색

원한다면 선형회귀나 Decision Tree 중 **하나만** 추가하거나, `GroupShuffleSplit`, 미지 ID별 RMSE, `permutation_importance` 가운데 하나만 골라 봅니다. 모든 항목을 완료할 필요는 없습니다. 같은 validation을 보며 모델이나 설정을 바꾸었다면 그 점수를 최종 test 성능이라고 부르지 않습니다.

## 여기까지 하면 이번 주 기록 완료

아래 네 줄을 이 셀 아래 새 Markdown 셀에 적습니다. 성공한 결과가 없어도 오류와 다음 행동을 적었다면 완료입니다.

- 질문 1개:
- 실행 또는 실행 시도 1개:
- 관찰한 결과 또는 오류 1개:
- 다음 행동 1개:
